# 🏠 House Price Prediction — Internship Project Week 1
**Dataset:** Housing Prices Dataset (Kaggle — 545 rows × 13 columns)  
**Goal:** Build regression models to predict house prices and identify key price drivers.


---
## Task 1 — Data Loading & Exploration

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
# Load the dataset
df = pd.read_csv('Housing.csv')

# Display first 10 rows
print("=== First 10 Rows ===")
print(df.head(10).to_string())

=== First 10 Rows ===
      price   area  bedrooms  bathrooms  stories mainroad guestroom basement hotwaterheating airconditioning  parking prefarea furnishingstatus
0   8826129   8920         3          3        3      yes        no       no             yes              no        1      yes      unfurnished
1   3285453   2510         3          2        2      yes       yes       no             yes              no        1       no        furnished
2   5880382   7040         4          1        2      yes        no       no              no             yes        2       no        furnished
3  10036525  15068         3          1        2      yes        no       no              no              no        3       no      unfurnished
4   5624496   6841         3          1        2      yes        no      yes              no              no        3       no      unfurnished
5   8956702  13614         5          2        1      yes       yes       no             yes              no      

In [3]:
# Shape of the dataset
print(f"Rows   : {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"\nColumn Names: {list(df.columns)}")

Rows   : 545
Columns: 13

Column Names: ['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']


In [4]:
# Target vs Features
print("TARGET column  : price")
print("FEATURE columns:", [c for c in df.columns if c != 'price'])
print("\nData Types:")
print(df.dtypes)

TARGET column  : price
FEATURE columns: ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']

Data Types:
price               int64
area                int64
bedrooms            int64
bathrooms           int64
stories             int64
mainroad              str
guestroom             str
basement              str
hotwaterheating       str
airconditioning       str
parking             int64
prefarea              str
furnishingstatus      str
dtype: object


In [5]:
# Check for missing values
print("=== Missing Values per Column ===")
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

=== Missing Values per Column ===
price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

Total missing values: 0


In [6]:
# Basic statistics
print("=== Dataset Statistics ===")
print(df.describe().to_string())

=== Dataset Statistics ===
              price          area    bedrooms   bathrooms     stories     parking
count  5.450000e+02    545.000000  545.000000  545.000000  545.000000  545.000000
mean   7.237890e+06   8815.911927    3.607339    1.961468    2.038532    1.238532
std    2.006152e+06   4137.463849    1.001810    0.838187    0.901582    0.902398
min    2.198610e+06   1654.000000    1.000000    1.000000    1.000000    0.000000
25%    5.769448e+06   5211.000000    3.000000    1.000000    1.000000    1.000000
50%    7.282624e+06   8672.000000    4.000000    2.000000    2.000000    1.000000
75%    8.611823e+06  12517.000000    4.000000    2.000000    3.000000    2.000000
max    1.330000e+07  16191.000000    6.000000    4.000000    4.000000    3.000000


---
## Task 2 — Data Cleaning

In [7]:
# Handle missing values (fill numeric with median, categorical with mode)
print("Missing values before:", df.isnull().sum().sum())

numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

object_cols = df.select_dtypes(include=['object']).columns
for col in object_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("Missing values after :", df.isnull().sum().sum())

Missing values before: 0
Missing values after : 0


In [8]:
# Remove duplicate rows
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f"Rows before: {before}  |  After removing duplicates: {after}")
print(f"Duplicates removed: {before - after}")

Rows before: 545  |  After removing duplicates: 545
Duplicates removed: 0


In [9]:
# Encode binary yes/no columns
binary_cols = ['mainroad','guestroom','basement','hotwaterheating','airconditioning','prefarea']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

# One-hot encode furnishingstatus (3 categories)
df = pd.get_dummies(df, columns=['furnishingstatus'], drop_first=True)

print("Columns after encoding:")
print(list(df.columns))
print("\nFinal dataset shape:", df.shape)

Columns after encoding:
['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus_semi-furnished', 'furnishingstatus_unfurnished']

Final dataset shape: (545, 14)


---
## Task 3 — Model Building

In [10]:
# Define features and target
X = df.drop('price', axis=1)
y = df['price']

print("Feature columns:", list(X.columns))
print(f"\nX shape: {X.shape}  |  y shape: {y.shape}")

Feature columns: ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus_semi-furnished', 'furnishingstatus_unfurnished']

X shape: (545, 13)  |  y shape: (545,)


In [11]:
# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training rows : {X_train.shape[0]}")
print(f"Test rows     : {X_test.shape[0]}")

Training rows : 436
Test rows     : 109


In [12]:
# === Model 1: Linear Regression ===
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)

print("=== Linear Regression ===")
print(f"  MAE  : {mae_lr:>12,.0f}")
print(f"  RMSE : {rmse_lr:>12,.0f}")
print(f"  R²   : {r2_lr:>12.4f}")

=== Linear Regression ===
  MAE  :      579,414
  RMSE :      727,287
  R²   :       0.8513


In [13]:
# === Model 2: Random Forest Regressor ===
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf   = r2_score(y_test, y_pred_rf)

print("=== Random Forest Regressor ===")
print(f"  MAE  : {mae_rf:>12,.0f}")
print(f"  RMSE : {rmse_rf:>12,.0f}")
print(f"  R²   : {r2_rf:>12.4f}")

=== Random Forest Regressor ===
  MAE  :      685,023
  RMSE :      833,430
  R²   :       0.8048


In [14]:
# === Model Comparison Table ===
comparison = pd.DataFrame({
    'Model'   : ['Linear Regression', 'Random Forest'],
    'MAE'     : [f'{mae_lr:,.0f}',  f'{mae_rf:,.0f}'],
    'RMSE'    : [f'{rmse_lr:,.0f}', f'{rmse_rf:,.0f}'],
    'R² Score': [f'{r2_lr:.4f}',   f'{r2_rf:.4f}'],
})
print(comparison.to_string(index=False))

            Model     MAE    RMSE R² Score
Linear Regression 579,414 727,287   0.8513
    Random Forest 685,023 833,430   0.8048


---
## Task 4 — Visualizations

In [15]:
# === Chart 1: Distribution of House Prices ===
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df['price'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(df['price'].mean(),   color='crimson', linestyle='--', linewidth=1.8, label=f"Mean: {df['price'].mean():,.0f}")
ax.axvline(df['price'].median(), color='orange',  linestyle='--', linewidth=1.8, label=f"Median: {df['price'].median():,.0f}")
ax.set_title('Distribution of House Prices', fontsize=15, fontweight='bold', pad=12)
ax.set_xlabel('Price', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
plt.savefig('charts/chart1_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print("Chart 1 saved.")

Chart 1 saved.


In [16]:
# === Chart 2: Correlation Heatmap ===
fig, ax = plt.subplots(figsize=(11, 8))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('charts/chart2_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print("Chart 2 saved.")

Chart 2 saved.


In [17]:
# === Chart 3: Actual vs Predicted (both models side-by-side) ===
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, y_pred, title, color in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Linear Regression', 'Random Forest'],
    ['steelblue', 'seagreen']
):
    ax.scatter(y_test, y_pred, alpha=0.55, color=color, edgecolors='white', linewidth=0.4, s=50)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.8, label='Perfect Prediction')
    ax.set_xlabel('Actual Price', fontsize=11)
    ax.set_ylabel('Predicted Price', fontsize=11)
    ax.set_title(f'Actual vs Predicted — {title}', fontsize=12, fontweight='bold')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
    r2 = r2_score(y_test, y_pred)
    ax.legend(title=f'R² = {r2:.3f}', fontsize=10)
plt.suptitle('Actual vs Predicted House Prices', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('charts/chart3_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print("Chart 3 saved.")

Chart 3 saved.


In [18]:
# === Bonus Chart 4: Feature Importance (Random Forest) ===
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#E91E63' if v >= feat_imp.quantile(0.7) else '#2196F3' for v in feat_imp]
feat_imp.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Feature Importances — Random Forest', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Importance Score', fontsize=11)
plt.tight_layout()
plt.savefig('charts/chart4_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print("Bonus chart saved.")

Bonus chart saved.


---
## Task 5 — Insights & Summary

### Key Findings

**Which features influence house price the most?**  
Based on the Random Forest feature importance scores, **area (square footage)** is by far the dominant driver of house price (importance ≈ 0.76), followed by **number of bathrooms** (≈ 0.06), **number of stories** (≈ 0.03), and **parking availability**. Amenity features like air conditioning and preferred area location also contribute, while guest room and hot water heating show comparatively lower impact.

**How accurate was the model (in plain terms)?**  
The **Linear Regression model achieved an R² of 0.85**, meaning it explains 85% of the variance in house prices — a strong result for a dataset this size. The Random Forest model scored R² ≈ 0.80. Interestingly, Linear Regression outperformed Random Forest here, suggesting that the price relationship with features is largely linear in this dataset, making simpler models very effective.

**What was surprising in the data?**  
It was surprising that **area alone accounted for ~76% of feature importance** in the Random Forest model — far outweighing all other features combined. Also, the price distribution is noticeably **right-skewed**, with a small number of very high-priced properties pulling the mean above the median. Furnishing status had a smaller effect than expected.

**Recommendation for a real estate business:**  
Focus investment on **maximizing usable floor area and the number of bathrooms** when developing or renovating properties — these yield the highest return in terms of price premium. For properties targeting premium buyers, adding **parking spaces and air conditioning** are high-ROI amenities. Cosmetic upgrades like furnishing have a comparatively smaller impact on market value.


In [19]:
# Final Summary
print("=" * 55)
print("     HOUSE PRICE PREDICTION — FINAL SUMMARY")
print("=" * 55)
print(f"Dataset shape  : {df.shape}")
print(f"Features used  : {X.shape[1]}")
print()
print("Linear Regression:")
print(f"  MAE   = {mae_lr:>12,.0f}")
print(f"  RMSE  = {rmse_lr:>12,.0f}")
print(f"  R²    = {r2_lr:>12.4f}  ← Best model")
print()
print("Random Forest Regressor:")
print(f"  MAE   = {mae_rf:>12,.0f}")
print(f"  RMSE  = {rmse_rf:>12,.0f}")
print(f"  R²    = {r2_rf:>12.4f}")
print()
print("Top 3 Most Important Features (Random Forest):")
top3 = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(3)
for i, (feat, score) in enumerate(top3.items(), 1):
    print(f"  {i}. {feat:30s} {score:.4f}")

     HOUSE PRICE PREDICTION — FINAL SUMMARY
Dataset shape  : (545, 14)
Features used  : 13

Linear Regression:
  MAE   =      579,414
  RMSE  =      727,287
  R²    =       0.8513  ← Best model

Random Forest Regressor:
  MAE   =      685,023
  RMSE  =      833,430
  R²    =       0.8048

Top 3 Most Important Features (Random Forest):
  1. area                           0.7595
  2. bathrooms                      0.0615
  3. stories                        0.0331
